# 04 — Machine Learning Forecasting & SHAP Regime Decomposition

## Research Objective
Can gradient-boosted trees with regime-aware features outperform classical time-series baselines (ARIMA/GARCH), and how does feature importance shift between normal and crisis market regimes?

We apply **LightGBM** with **SHAP (SHapley Additive exPlanations)** to attribute return and volatility predictions to physical storage levels, 5-year inventory deviations, winter premiums, and seasonal harmonics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.models.gbm_model import run_gbm_model
from src.models.evaluation import ModelEvaluator

df_features = pd.read_parquet('data/processed/feature_matrix.parquet')
gbm, explanation = run_gbm_model(df=df_features, target_col='target_return_5d')

comparison_table = explanation['decomposed_importance']['comparison']
print('Top Features Ranked by Crisis Regime SHAP Impact:')
display(comparison_table.head(10))

### 4.1 Stratified SHAP Decomposition Plot
Observe the massive escalation in importance for physical storage deviation and winter deadlines during crisis periods:

In [ ]:
from src.visualization.plots import plot_shap_regime_decomposition
plot_shap_regime_decomposition(comparison_table)

### 4.2 Expanding-Window Cross-Validation vs ARIMA Baseline
Rigorous time-series forward-chaining evaluation:

In [ ]:
evaluator = ModelEvaluator(df_features)
cv_results = evaluator.evaluate_expanding_window(target_col='target_return_5d')
for k, v in cv_results.items():
    print(f'{k:<30}: {v}')